In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# טכניקות לצמצום ודיכוי שגיאות

> **Note:** גרסת הבטא של מודל הרצה חדש זמינה כעת. מודל ההרצה הממוקד מספק גמישות רבה יותר בהתאמה אישית של תהליך עבודת צמצום השגיאות שלך. ראה את המדריך [מודל הרצה ממוקד](/guides/directed-execution-model) לפרטים נוספים.


<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
טכניקות לצמצום שגיאות ולדיכוי שגיאות משמשות לשיפור איכות התוצאות בעת ​​הגדלת עומסי עבודה. עמוד זה מספק הסברים ברמה גבוהה על טכניקות דיכוי השגיאות וצמצום השגיאות הזמינות דרך Qiskit Runtime.

תא הקוד הבא מייבא את ה-primitive של Estimator ויוצר Backend שישמש לאתחול ה-Estimator בתאי קוד מאוחרים יותר.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## ניתוק דינמי
מעגלי Quantum מורצים על חומרת IBM&reg; כרצפים של פולסי מיקרוגל הדורשים תזמון והרצה במרווחי זמן מדויקים.
למרבה הצער, אינטראקציות לא רצויות בין Qubits עלולות להוביל לשגיאות קוהרנטיות ב-Qubits שנמצאים בסרק. הניתוק הדינמי פועל על ידי הכנסת רצפי פולסים ל-Qubits סרקים כדי לבטל בקירוב את ההשפעה של שגיאות אלו. כל רצף פולסים שמוכנס שווה ערך לפעולת זהות, אך הנוכחות הפיזית של הפולסים מדכאת שגיאות.
קיימות בחירות רבות אפשריות של רצפי פולסים, ואיזה רצף עדיף לכל מקרה פרטי נותר [תחום מחקר פעיל](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

שים לב שניתוק דינמי שימושי בעיקר עבור מעגלים המכילים פערים שבהם חלק מה-Qubits יושבים בסרק ללא פעולות כלשהן שפועלות עליהם. אם הפעולות במעגל ארוזות בצפיפות רבה, כך שכל ה-Qubits עסוקים רוב הזמן, אזי הוספת פולסי ניתוק דינמי עשויה לא לשפר את הביצועים. למעשה, היא אף עלולה להרע את הביצועים עקב אי-שלמויות בפולסים עצמם.

התרשים שלהלן מתאר ניתוק דינמי עם רצף פולסים XX. ה-Circuit המופשט משמאל ממופה לתזמון פולס מיקרוגל בפינה הימנית העליונה. הפינה הימנית התחתונה מתארת את אותו תזמון, אך עם רצף של שני פולסי X המוכנסים במהלך תקופת סרק של ה-Qubit הראשון.

![Depiction of dynamical decoupling](../docs/images/guides/error-mitigation-explanation/dd.avif)

ניתן להפעיל ניתוק דינמי על ידי הגדרת `enable` ל-`True` ב[אפשרויות הניתוק הדינמי](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). ניתן להשתמש באפשרות `sequence_type` כדי לבחור מבין מספר רצפי פולסים שונים. סוג הרצף המוגדר כברירת מחדל הוא `"XX"`.

תא הקוד הבא מראה כיצד להפעיל ניתוק דינמי עבור ה-Estimator ולבחור רצף ניתוק דינמי.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## סיבוב פאולי
סיבוב (Twirling), הידוע גם בשם [קימפול אקראי](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), הוא טכניקה נפוצה להמרת ערוצי רעש שרירותיים לערוצי רעש בעלי מבנה ספציפי יותר.

סיבוב פאולי הוא סוג מיוחד של סיבוב המשתמש בפעולות פאולי. יש לו את האפקט של המרת כל ערוץ קוונטי לערוץ פאולי. כשמבצעים אותו לבד, הוא יכול לצמצם רעש קוהרנטי מכיוון שרעש קוהרנטי נוטה להצטבר בריבועית עם מספר הפעולות, בעוד רעש פאולי מצטבר באופן ליניארי. סיבוב פאולי משולב לעיתים קרובות עם טכניקות אחרות לצמצום שגיאות שעובדות טוב יותר עם רעש פאולי מאשר עם רעש שרירותי.

סיבוב פאולי מיושם על ידי עטיפת קבוצה נבחרת של Gate-ים עם Gate-ים אקראיים של Pauli חד-Qubit באופן שהאפקט האידיאלי של ה-Gate נשאר זהה. התוצאה היא שמעגל בודד מוחלף במכלול אקראי של מעגלים, כולם בעלי אותו אפקט אידיאלי. בעת דגימת המעגל, דגימות נלקחות ממספר מופעים אקראיים, ולא רק מאחד.

![Depiction of Pauli twirling](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

מכיוון שרוב השגיאות בחומרה קוונטית נוכחית מגיעות מ-Gate-ים של שני Qubits, טכניקה זו מוחלת לרוב באופן בלעדי על Gate-ים (מקוריים) של שני Qubits. התרשים הבא מתאר כמה סיבובי פאולי עבור ה-Gate-ים CNOT ו-ECR. כל מעגל בשורה הוא בעל אותו אפקט אידיאלי.

![Depiction of gate twirls](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

ניתן להפעיל סיבוב פאולי על ידי הגדרת `enable_gates` ל-`True` ב[אפשרויות הסיבוב](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). אפשרויות בולטות נוספות כוללות:

- `num_randomizations`: מספר מופעי המעגל לשלוף מהמכלול של מעגלים מסובבים.
- `shots_per_randomization`: מספר ה-shots לדגום מכל מופע מעגל.

תא הקוד הבא מראה כיצד להפעיל סיבוב פאולי ולהגדיר אפשרויות אלה עבור ה-Estimator. אין צורך להגדיר באופן מפורש אף אחת מהאפשרויות הללו.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## ביטול שגיאות מדידה מסובבת (TREX)
[ביטול שגיאות מדידה מסובבת (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) מצמצם את השפעת שגיאות המדידה עבור הערכת ערכי ציפייה של תצפיות פאולי.
הוא מבוסס על המושג של מדידות מסובבות, המושגות על ידי החלפה אקראית של Gate-ים של מדידה ברצף של (1) Gate X של פאולי, (2) מדידה, ו-(3) היפוך סיביות קלאסי. בדיוק כמו בסיבוב Gate רגיל, רצף זה שווה ערך למדידה פשוטה בהיעדר רעש, כפי שמתואר בתרשים הבא:

![Depiction of measurement twirling](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

בנוכחות שגיאת מדידה, לסיבוב המדידה יש ​​את האפקט של אלכסון מטריצת העברת שגיאות המדידה, מה שמקל על ההיפוך שלה. הערכת מטריצת העברת שגיאות המדידה דורשת הרצת מעגלי כיול נוספים, מה שמביא לתקורה קטנה.

ניתן להפעיל TREX על ידי הגדרת `measure_mitigation` ל-`True` ב[אפשרויות העמידות של Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) עבור ה-Estimator. האפשרויות ללמידת רעש מדידה מתוארות [כאן](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). כמו בסיבוב Gate, ניתן להגדיר את מספר האקראות של המעגל ואת מספר ה-shots לכל אקראות.

תא הקוד הבא מראה כיצד להפעיל TREX ולהגדיר אפשרויות אלה עבור ה-Estimator. אין צורך להגדיר באופן מפורש אף אחת מהאפשרויות הללו.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## אקסטרפולציה לאפס רעש (ZNE)
אקסטרפולציה לאפס רעש (ZNE) היא טכניקה לצמצום שגיאות בהערכת ערכי ציפייה של תצפיות. למרות שהיא לעיתים קרובות משפרת תוצאות, אין ערובה שהיא תפיק תוצאה בלתי מוטה.

ZNE מורכבת משני שלבים:

1. _הגברת רעש_: המעגל הקוונטי המקורי מורץ מספר פעמים בשיעורי רעש שונים.
2. _אקסטרפולציה_: התוצאה האידיאלית מוערכת על ידי אקסטרפולציה של תוצאות ערך הציפייה הרועשות לגבול האפס-רעש.

ניתן לממש הן את שלב הגברת הרעש והן את שלב האקסטרפולציה בדרכים שונות רבות. Qiskit Runtime מממש הגברת רעש על ידי "קיפול Gate דיגיטלי", מה שאומר ש-Gate-ים של שני Qubits מוחלפים ברצפים שקולים של ה-Gate והופכו. לדוגמה, החלפת יחידה $U$ ב-$U U^\dagger U$ תניב גורם הגברת רעש של 3. עבור האקסטרפולציה, ניתן לבחור מבין מספר צורות פונקציונליות, כולל התאמה ליניארית או התאמה מעריכית.
התמונה שלהלן מתארת ​​קיפול Gate דיגיטלי משמאל, ואת תהליך האקסטרפולציה מימין.

![Depiction of ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

ניתן להפעיל ZNE על ידי הגדרת `zne_mitigation` ל-`True` ב[אפשרויות העמידות של Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) עבור ה-Estimator.
האפשרויות של Qiskit Runtime ל-ZNE מתוארות [כאן](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). האפשרויות הבולטות הבאות:

- `noise_factors`: גורמי הרעש לשימוש עבור הגברת הרעש.
- `extrapolator`: הצורה הפונקציונלית לשימוש עבור האקסטרפולציה.

תא הקוד הבא מראה כיצד להפעיל ZNE ולהגדיר אפשרויות אלה עבור ה-Estimator. אין צורך להגדיר באופן מפורש אף אחת מהאפשרויות הללו.

In [5]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 1})

<span id="pea"></span>
## הגברת שגיאות הסתברותית (PEA)
אחד האתגרים העיקריים ב-ZNE הוא להגביר במדויק את הרעש המשפיע על המעגל המטרה. קיפול Gate מספק דרך קלה לביצוע הגברה זו, אך הוא עלול להיות לא מדויק ועלול להוביל לתוצאות שגויות. ראה את המאמר ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), ובפרט עמוד 4 של המידע המשלים לפרטים. הגברת שגיאות הסתברותית מספקת גישה מדויקת יותר להגברת שגיאות באמצעות למידת רעש.

PEA היא טכניקה מתוחכמת יותר המבצעת ניסויים מקדימים לשחזור הרעש ולאחר מכן משתמשת במידע זה לביצוע הגברה מדויקת. היא מתחילה בלמידת מודל הרעש המסובב של כל שכבה של Gate-ים שזירה במעגל לפני הרצתם (ראה [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) לאפשרויות למידה רלוונטיות). לאחר שלב הלמידה, המעגלים מורצים בכל גורם רעש, כאשר כל שכבת שזירה של המעגלים מוגברת על ידי הזרקה הסתברותית של רעש חד-Qubit פרופורציונלי למודל הרעש הנלמד המתאים. ראה את המאמר ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) לפרטים נוספים.

PEA מורכבת משלושה שלבים:
1. _למידה_: מודל הרעש המסובב של כל שכבה של Gate-ים שזירה במעגל נלמד.
1. _הגברת רעש_: המעגל הקוונטי המקורי מורץ מספר פעמים בגורמי רעש שונים.
2. _אקסטרפולציה_: התוצאה האידיאלית מוערכת על ידי אקסטרפולציה של תוצאות ערך הציפייה הרועשות לגבול האפס-רעש.

עבור ניסויים בקנה מידה שימושיות, PEA היא לרוב הבחירה הטובה ביותר.

מכיוון ש-PEA היא טכניקת הגברת רעש של ZNE, עליך גם להפעיל ZNE על ידי הגדרת `resilience.zne_mitigation = True`. ניתן גם להשתמש באפשרויות [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) אחרות להגדרת אקסטרפולטורים, רמות הגברה וכן הלאה. PEA דורשת מודל רעש, הנוצר אוטומטית בעת שימוש ב-primitives.

הקטע הבא מספק דוגמה שבה PEA משמשת לצמצום תוצאת עבודת Estimator:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pec"></span>
## ביטול שגיאות הסתברותי (PEC)
ביטול שגיאות הסתברותי (PEC) הוא טכניקה לצמצום שגיאות בהערכת ערכי ציפייה של תצפיות. בניגוד ל-ZNE, הוא מחזיר אומד בלתי מוטה של ​​ערך הציפייה. עם זאת, הוא בדרך כלל גורר תקורה גדולה יותר.

ב-PEC, האפקט של מעגל מטרה אידיאלי מבוטא כצירוף לינארי של מעגלים רועשים שניתנים למימוש בפועל:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

ניתן לשחזר את הפלט של המעגל האידיאלי על ידי הרצת מופעי מעגל רועשים שונים הנשלפים ממכלול אקראי המוגדר על ידי הצירוף הלינארי. אם המקדמים $\eta_i$ יוצרים התפלגות הסתברות, ניתן להשתמש בהם ישירות כהסתברויות של המכלול. בפועל, חלק מהמקדמים שליליים, ולכן הם יוצרים התפלגות מעין-הסתברות במקום. ניתן עדיין להשתמש בהם להגדרת מכלול אקראי, אך קיימת תקורת דגימה הקשורה לשליליות של התפלגות המעין-הסתברות, המאופיינת בכמות

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

תקורת הדגימה היא גורם כפלי על מספר ה-shots הנדרש להערכת ערך ציפייה לדיוק נתון, בהשוואה למספר ה-shots שיידרשו מהמעגל האידיאלי. הוא גדל ריבועית עם $\gamma$, אשר בתורו גדל מעריכית עם עומק המעגל.

ניתן להפעיל PEC על ידי הגדרת `pec_mitigation` ל-`True` ב[אפשרויות העמידות של Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) עבור ה-Estimator.
האפשרויות של Qiskit Runtime ל-PEC מתוארות [כאן](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). ניתן להגדיר מגבלה על תקורת הדגימה באמצעות אפשרות `max_overhead`. שים לב שהגבלת תקורת הדגימה עלולה לגרום לדיוק התוצאה לחרוג מהדיוק המבוקש. ערך ברירת המחדל של `max_overhead` הוא 100.

תא הקוד הבא מראה כיצד להפעיל PEC ולהגדיר את אפשרות `max_overhead` עבור ה-Estimator.

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

## השלבים הבאים
- ראה ב[מדריך](/tutorials/combine-error-mitigation-techniques) על שילוב אפשרויות צמצום שגיאות עם ה-primitive של Estimator.
- [הגדר צמצום שגיאות.](configure-error-mitigation)
- [הגדר דיכוי שגיאות.](configure-error-suppression)
- חקור [אפשרויות](runtime-options-overview) נוספות עבור ה-primitives של Qiskit Runtime.
- החלט באיזה [מצב הרצה](execution-modes) להריץ את העבודה שלך.

In [8]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Probabilistic error cancellation (PEC)

Probabilistic error cancellation (PEC) is a technique for mitigating errors in estimating expectation values of observables. Unlike ZNE, it returns an unbiased estimate of the expectation value. However, it generally incurs a greater overhead.

In PEC, the effect of an ideal target circuit is expressed as a linear combination of noisy circuits that are actually implementable in practice:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

The output of the ideal circuit can then be reproduced by executing different noisy circuit instances drawn from a random ensemble defined by the linear combination. If the coefficients $\eta_i$ form a probability distribution, they can be used directly as the probabilities of the ensemble. In practice, some of the coefficients are negative, so they form a quasi-probability distribution instead. They can still be used to define a random ensemble, but there is a sampling overhead related to the negativity of the quasi-probability distribution, which is characterized by the quantity

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

The sampling overhead is a multiplicative factor on the number of shots required to estimate an expectation value to a given precision, compared to the number of shots that would be needed from the ideal circuit. It scales quadratically with $\gamma$, which in turn scales exponentially with the depth of the circuit.

PEC can be enabled by setting `pec_mitigation` to `True` in the [Qiskit Runtime resilience options](/docs/api/qiskit-ibm-runtime/options-resilience-options-v2) for Estimator.
The Qiskit Runtime options for PEC are described [here](/docs/api/qiskit-ibm-runtime/options-pec-options). A limit on the sampling overhead can be set using the `max_overhead` option. Note that limiting the sampling overhead can cause the precision of the result to exceed the requested precision. The default value of `max_overhead` is 100.

The following code cell shows how to enable PEC and set the `max_overhead` option for Estimator.

In [9]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Next steps

- Check out the [tutorial](/docs/tutorials/combine-error-mitigation-techniques) on combining error mitigation options with the Estimator primitive.
- [Configure noise management with Estimator](/docs/guides/estimator-noise-management).
- [Configure noise management with Sampler](/docs/guides/sampler-noise-management).
- Explore other [options](/docs/guides/runtime-options-overview) for the Qiskit Runtime primitives.
- Decide what [execution mode](/docs/guides/execution-modes) to run your job in.